A dokumentációba meg kell említeni, hogy a kernelt át kell állítani az ArcPy-osra, plusz a mappákat átírni, kommentelni, meg beletenni a dokumentációba 

In [4]:
import arcpy
import os
import re
from datetime import datetime

gdb = r"C:\Users\emese\Documents\ErasmusGIS\thesis\Results_copy.gdb"
out_folder = r"C:\Users\emese\Documents\ErasmusGIS\thesis\Data\EuropeanDroughtObservatory\Data_dated_py"

arcpy.env.workspace = gdb
arcpy.env.overwriteOutput = True

In [5]:
fcs = arcpy.ListFeatureClasses()

print("total features:", len(fcs))

for fc in fcs:
    print(fc)

total features: 352
cdinx_m_edo_20200501_t_400_z04point
cdinx_m_edo_20200501_t_400_z04poli
cdinx_m_edo_20200501_t_400_z04smooth
cdinx_m_edo_20200501_t_400_z04final
cdinx_m_edo_20200511_t_400_z04point
cdinx_m_edo_20200511_t_400_z04poli
cdinx_m_edo_20200511_t_400_z04smooth
cdinx_m_edo_20200511_t_400_z04final
cdinx_m_edo_20200521_t_400_z04point
cdinx_m_edo_20200521_t_400_z04poli
cdinx_m_edo_20200521_t_400_z04smooth
cdinx_m_edo_20200521_t_400_z04final
cdinx_m_edo_20200601_t_400_z04point
cdinx_m_edo_20200601_t_400_z04poli
cdinx_m_edo_20200601_t_400_z04smooth
cdinx_m_edo_20200601_t_400_z04final
cdinx_m_edo_20200611_t_400_z04point
cdinx_m_edo_20200611_t_400_z04poli
cdinx_m_edo_20200611_t_400_z04smooth
cdinx_m_edo_20200611_t_400_z04final
cdinx_m_edo_20200621_t_400_z04point
cdinx_m_edo_20200621_t_400_z04poli
cdinx_m_edo_20200621_t_400_z04smooth
cdinx_m_edo_20200621_t_400_z04final
cdinx_m_edo_20200701_t_400_z04point
cdinx_m_edo_20200701_t_400_z04poli
cdinx_m_edo_20200701_t_400_z04smooth
cdinx_m_

In [8]:
for fc in fcs:

    if "final" not in fc.lower():
        continue

    print("Feldolgozás:", fc)

    match = re.search(r'(\d{8})', fc)
    if not match:
        continue

    date_str = match.group(1)
    date_obj = datetime.strptime(date_str, "%Y%m%d")

    # date mező
    fields = [f.name for f in arcpy.ListFields(fc)]

    if "date" not in fields:
        arcpy.AddField_management(fc, "date", "DATE")

    expr = f"datetime({date_obj.year}, {date_obj.month}, {date_obj.day})"

    arcpy.CalculateField_management(
        fc,
        "date",
        expr,
        "PYTHON3"
        )

    # SHP export
    out_name = f"EDO_Hun_{date_str}.shp"
    out_path = os.path.join(out_folder, out_name)

    arcpy.CopyFeatures_management(fc, out_path)

    print("  -> export kész:", out_name)

print("Files ready.")

Feldolgozás: cdinx_m_edo_20200501_t_400_z04final
  -> export kész: EDO_Hun_20200501.shp
Feldolgozás: cdinx_m_edo_20200511_t_400_z04final
  -> export kész: EDO_Hun_20200511.shp
Feldolgozás: cdinx_m_edo_20200521_t_400_z04final
  -> export kész: EDO_Hun_20200521.shp
Feldolgozás: cdinx_m_edo_20200601_t_400_z04final
  -> export kész: EDO_Hun_20200601.shp
Feldolgozás: cdinx_m_edo_20200611_t_400_z04final
  -> export kész: EDO_Hun_20200611.shp
Feldolgozás: cdinx_m_edo_20200621_t_400_z04final
  -> export kész: EDO_Hun_20200621.shp
Feldolgozás: cdinx_m_edo_20200701_t_400_z04final
  -> export kész: EDO_Hun_20200701.shp
Feldolgozás: cdinx_m_edo_20200711_t_400_z04final
  -> export kész: EDO_Hun_20200711.shp
Feldolgozás: cdinx_m_edo_20200721_t_400_z04final
  -> export kész: EDO_Hun_20200721.shp
Feldolgozás: cdinx_m_edo_20200801_t_400_z04final
  -> export kész: EDO_Hun_20200801.shp
Feldolgozás: cdinx_m_edo_20200811_t_400_z04final
  -> export kész: EDO_Hun_20200811.shp
Feldolgozás: cdinx_m_edo_2020082

In [9]:
import glob

shps = glob.glob(os.path.join(out_folder, "EDO_Hun_*.shp"))

for shp in shps:

    print("Dissolve:", shp)

    out_name = shp.replace(".shp", "_diss.shp")
    out_path = os.path.join(out_folder, out_name)

    arcpy.Dissolve_management(
        shp,
        out_path,
        ["gridcode", "date"]
    )

    print("  -> kész:", out_name)

print("Dissolve done.")

Dissolve: C:\Users\emese\Documents\ErasmusGIS\thesis\Data\EuropeanDroughtObservatory\Data_dated_py\EDO_Hun_20200501.shp
  -> kész: C:\Users\emese\Documents\ErasmusGIS\thesis\Data\EuropeanDroughtObservatory\Data_dated_py\EDO_Hun_20200501_diss.shp
Dissolve: C:\Users\emese\Documents\ErasmusGIS\thesis\Data\EuropeanDroughtObservatory\Data_dated_py\EDO_Hun_20200511.shp
  -> kész: C:\Users\emese\Documents\ErasmusGIS\thesis\Data\EuropeanDroughtObservatory\Data_dated_py\EDO_Hun_20200511_diss.shp
Dissolve: C:\Users\emese\Documents\ErasmusGIS\thesis\Data\EuropeanDroughtObservatory\Data_dated_py\EDO_Hun_20200521.shp
  -> kész: C:\Users\emese\Documents\ErasmusGIS\thesis\Data\EuropeanDroughtObservatory\Data_dated_py\EDO_Hun_20200521_diss.shp
Dissolve: C:\Users\emese\Documents\ErasmusGIS\thesis\Data\EuropeanDroughtObservatory\Data_dated_py\EDO_Hun_20200601.shp
  -> kész: C:\Users\emese\Documents\ErasmusGIS\thesis\Data\EuropeanDroughtObservatory\Data_dated_py\EDO_Hun_20200601_diss.shp
Dissolve: C:\Use